[Dados Abertos - CNPJ](https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9)


[2024-02](https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9?dir=/2024-02)

[2025-02](https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9?dir=/2025-02)

[2026-02](https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9?dir=/2026-02)

Arquivos `Estabelecimentos1.zip`


In [1]:
%pip install pyspark duckdb


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from IPython.core.magic import (register_line_magic, register_cell_magic,register_line_cell_magic)

@register_line_cell_magic('sql')
def sparksql(line, cell=None):
    "Esse Magic funciona com  %sql e %%sql"
    try: spark
    except NameError: print('Spark instance is not defined')
    else:
        spark.conf.set('spark.sql.repl.eagerEval.enabled','true')
        spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 100)
        spark.conf.set('spark.sql.repl.eagerEval.truncate',-1)
        if cell is None:
            result = spark.sql(line)
            return result
        else:
            result = spark.sql(cell)
            return result

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import time


Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","10G")
Config.set("spark.memory.fraction", 0.9)
Config.set("spark.sql.adaptive.enabled", "true")
Config.set("spark.sql.adaptive.join.enabled", "true")
Config.set('spark.sql.legacy.allowNonEmptyLocationInCTAS','true')

# Config.set("spark.sql.shuffle.partitions", 100)
# Config.set("spark.default.parallelism", 200)


spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("CNPJ_Pipeline").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 19:03:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 19:03:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### https://spark.apache.org/docs/latest/sql-ref-datatypes.html

*   withColumn
*   try_cast
*   to_date
*   lit
*   Int x Long
*   UDF
*   fillna

## Construindo o Pipeline



33.683.111/0001-07


33683111000107

In [ ]:
!mkdir -p "/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Aula 04/Data"

In [15]:
path = "/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Aula 04/Estabelecimentos1_2024-02"

df = (
    spark.read
    .option("sep", ";")
    .option("quote", '"')
    .option("header", "false")
    .option("encoding", "ISO-8859-1")
    .csv(path)
)

In [16]:
df.repartition(1).write.parquet("raw_data/")

In [ ]:
_version_file = '2024-02'

df_raw = spark.read.parquet(f'/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Aula 04/Estabelecimentos1_{_version_file}.parquet')

df_raw = df_raw.select(*[nullif(df_raw[column_name], lit('')).alias(column_name) for column_name in df_raw.columns])

df_raw = df_raw.withColumn("CNPJ", concat(df_raw["CNPJ_BASICO"],df_raw["CNPJ_ORDEM"],df_raw["CNPJ_DV"]))
df_raw = df_raw.withColumn("version_file", lit(_version_file))


df_raw = df_raw.select(
    df_raw["CNPJ"].try_cast(LongType()).alias("CNPJ"),
    df_raw["CNPJ_BASICO"].try_cast(IntegerType()).alias("CNPJ_BASICO"),
    df_raw["CNPJ_ORDEM"].try_cast(IntegerType()).alias("CNPJ_ORDEM"),
    df_raw["CNPJ_DV"].try_cast(IntegerType()).alias("CNPJ_DV"),
    df_raw["MATRIZ_FILIAL"].try_cast(IntegerType()).alias("MATRIZ_FILIAL"),
    df_raw["NOME_FANTASIA"].try_cast(StringType()).alias("NOME_FANTASIA"),
    df_raw["SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("SITUACAO_CADASTRAL"),

    to_date(nullif(df_raw["DATA_SITUACAO_CADASTRAL"],lit('0')), "yyyyMMdd").alias("DATA_SITUACAO_CADASTRAL"),

    df_raw["MOTIVO_SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("MOTIVO_SITUACAO_CADASTRAL"),
    df_raw["CIDADE_EXTERIOR"].try_cast(StringType()).alias("CIDADE_EXTERIOR"),
    df_raw["PAIS"].try_cast(StringType()).alias("PAIS"),

    to_date(df_raw["DATA_DE_INICIO_ATIVIDADE"], "yyyyMMdd").alias("DATA_DE_INICIO_ATIVIDADE"),

    df_raw["CNAE_PRINCIPAL"].try_cast(IntegerType()).alias("CNAE_PRINCIPAL"),

    split(df_raw["CNAE_SECUNDARIO"],',').alias("CNAE_SECUNDARIO"),

    df_raw["TIPO_LOGRADOURO"].try_cast(StringType()).alias("TIPO_LOGRADOURO"),
    df_raw["LOGRADOURO"].try_cast(StringType()).alias("LOGRADOURO"),
    df_raw["NUMERO"].try_cast(StringType()).alias("NUMERO"),
    df_raw["COMPLEMENTO"].try_cast(StringType()).alias("COMPLEMENTO"),
    df_raw["BAIRRO"].try_cast(StringType()).alias("BAIRRO"),
    df_raw["CEP"].try_cast(StringType()).alias("CEP"),
    df_raw["UF"].try_cast(StringType()).alias("UF"),
    df_raw["MUNICIPIO"].try_cast(StringType()).alias("MUNICIPIO"),
    df_raw["DDD"].try_cast(StringType()).alias("DDD"),
    df_raw["TELEFONE"].try_cast(StringType()).alias("TELEFONE"),
    df_raw["DDD_2"].try_cast(StringType()).alias("DDD_2"),
    df_raw["TELEFONE_2"].try_cast(StringType()).alias("TELEFONE_2"),
    df_raw["DDD_FAX"].try_cast(StringType()).alias("DDD_FAX"),
    df_raw["FAX"].try_cast(StringType()).alias("FAX"),
    df_raw["CORREIO_ELETRONICO"].try_cast(StringType()).alias("CORREIO_ELETRONICO"),
    df_raw["SITUACAO_ESPECIAL"].try_cast(StringType()).alias("SITUACAO_ESPECIAL"),
    df_raw["DATA_SITUACAO_ESPECIAL"].try_cast(StringType()).alias("DATA_SITUACAO_ESPECIAL"),
    df_raw["version_file"].try_cast(StringType()).alias("version_file"),
    current_timestamp().alias("update_date")
)
df_raw.createOrReplaceTempView("raw_data")


df_raw = df_raw.na.fill({'CNPJ': -1, 'CNPJ_BASICO': -1, 'CNPJ_ORDEM': -1, 'CNPJ_DV': -1, 'MATRIZ_FILIAL': -1,'NOME_FANTASIA': 'N/A', 'SITUACAO_CADASTRAL': -1})

df_raw

#df_raw.repartition(1).write.partitionBy("UF").parquet(f"Data/trusted_data", mode='overwrite')




CNPJ,CNPJ_BASICO,CNPJ_ORDEM,CNPJ_DV,MATRIZ_FILIAL,NOME_FANTASIA,SITUACAO_CADASTRAL,DATA_SITUACAO_CADASTRAL,MOTIVO_SITUACAO_CADASTRAL,CIDADE_EXTERIOR,PAIS,DATA_DE_INICIO_ATIVIDADE,CNAE_PRINCIPAL,CNAE_SECUNDARIO,TIPO_LOGRADOURO,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CEP,UF,MUNICIPIO,DDD,TELEFONE,DDD_2,TELEFONE_2,DDD_FAX,FAX,CORREIO_ELETRONICO,SITUACAO_ESPECIAL,DATA_SITUACAO_ESPECIAL,version_file,update_date
7396865000168,7396865,1,68,1,N/A,8,2017-02-10,1,NULL,NULL,2005-05-18,1412602,[1411801],RUA,TUCANEIRA,30,NULL,DOS LAGOS,89136000,SC,8297,47,33851125,47,33851125,47,33851125,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
64904295001851,64904295,18,51,2,N/A,8,2016-11-10,1,NULL,NULL,2005-04-29,4639701,[4637199],AVENIDA,MENINO MARCELO,8551B,LOTE 2 QUADRAF,SERRARIA,57046000,AL,2785,11,36491000,31,33880436,82,33118379,CLAUDIO.GIGLIO@CAMIL.COM.BR,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
76016369000316,76016369,3,16,2,N/A,4,2023-11-13,63,NULL,NULL,1985-12-12,4632001,NULL,RUA,DO COMERCIO,55,SALA 7 GALERIA,CENTRO,11010141,SP,7071,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
52302726000182,52302726,1,82,1,N/A,4,2021-04-06,63,NULL,NULL,1983-02-23,4712100,NULL,RUA,GREGORIO LUCHIARI,496,NULL,SAO VITO,13472080,SP,6131,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
7396923000153,7396923,1,53,1,N/A,8,2014-01-15,1,NULL,NULL,2005-05-16,4721102,NULL,RUA,DA MOOCA,3336,NULL,MOOCA,03165000,SP,7107,011,69658088,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
3650261000145,3650261,1,45,1,OTICA PERFEICAO,4,2019-03-22,63,NULL,NULL,1999-12-17,4783101,[4783102],RUA,PREFEITO JOAO ORESTES DE ARAUJO,541,LOJA 03,CENTRO,88495000,SC,8113,048,2423953,NULL,NULL,048,2423953,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
7396929000120,7396929,1,20,1,N/A,3,2011-11-07,21,NULL,NULL,2005-05-04,6201501,[6204000],RUA,"FLORIANO PEIXOTO,",85,NULL,SANTA PAULA,09541350,SP,7077,11,32281722,NULL,NULL,11,32281722,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
25040718000132,25040718,1,32,1,COOCULTURA LTDA,8,2005-05-23,1,NULL,NULL,1991-08-23,6424701,NULL,RUA,ROSULINO FERREIRA GUIMA,767,NULL,CENTRO,75902261,GO,9571,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
7396936000122,7396936,1,22,1,N/A,8,2005-12-19,1,NULL,NULL,2005-05-25,4511102,NULL,AVENIDA,DOM PEDRO II,1748,NULL,CARLOS PRATES,30710010,MG,4123,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226
7396943000124,7396943,1,24,1,MEGA TELECOM,8,2017-02-17,1,NULL,NULL,2005-05-24,4752100,[9512600],AVENIDA,PARANA,465,NULL,CENTRO,83800000,PR,7679,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-05-25 19:28:10.391226


26/05/25 21:56:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 933185 ms exceeds timeout 120000 ms
26/05/25 21:56:49 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/25 21:56:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$



---



## Por que é uma boa prática sempre fazer um tratamento de Nulos em uma camara Trusted?

In [ ]:
%%sql
WITH Empresas (CNPJ, NOME_FANTASIA, SITUACAO_CADASTRAL)
AS
(
    VALUES
    (3650261000145, 'OTICA PERFEICAO', 4),
    (25040718000132, 'COOCULTURA LTDA', 8),
    (7396943000124, 'MEGA TELECOM', Null)
)
SELECT *
FROM Empresas where SITUACAO_CADASTRAL not in (8)





---



## UDF's -> Ajudam demais, mas evitem usar

*   Flexibilidade: Essenciais para transformações específicas, limpezas de dados complexas, criptografia ou análise personalizada que as funções nativas (ex: sum, avg) não cobrem.

*   Desempenho: UDFs tradicionais em Python (PySpark) podem ser lentas pois funcionam como "caixas-pretas" para o otimizador Catalyst, sendo processadas linha a linha.


*   Uso: Recomendadas para manipulações de dados que não podem ser expressas em SQL, úteis em operações ad-hoc, mas devem ser evitadas se uma função nativa do Spark puder resolver o problema de forma mais performática.

In [ ]:
def format_cnpj(cnpj_basico, cnpj_ordem, cnpj_dv):
  _cnpj_basico=str(cnpj_basico).zfill(8)
  _cnpj_ordem=str(cnpj_ordem).zfill(4)
  _cnpj_dv=str(cnpj_dv).zfill(2)
  return f'{_cnpj_basico[:2]}.{_cnpj_basico[2:5]}.{_cnpj_basico[5:8]}/{_cnpj_ordem.zfill(4)}-{_cnpj_dv.zfill(2)}'


format_cnpj(3650261,1,45)

'03.650.261/0001-45'

In [ ]:
## quando for executar com withcolumn
format_cnpj_udf = udf(format_cnpj, StringType())

## quando for executar no Spark SQL
spark.udf.register("format_cnpj_udf", format_cnpj, StringType())


<function __main__.format_cnpj(cnpj_basico, cnpj_ordem, cnpj_dv)>

In [ ]:
%%sql
select format_cnpj_udf(CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV) as CNPJ_FORMATADO from raw_data limit 2

CNPJ_FORMATADO
07.396.865/0001-68
64.904.295/0018-51




---



### Utilizando o Spark-submit

O que significa:

*   Config.set("spark.sql.adaptive.enabled", "true")
*   Config.set("spark.sql.adaptive.join.enabled", "true")
*   Config.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
*   Config.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
*   Config.set("spark.sql.sources.partitionOverwriteMode","dynamic")




In [ ]:
%lsmagic

In [ ]:
%%writefile empresas.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import time

Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","10G")
Config.set("spark.memory.fraction", 0.9)
Config.set("spark.sql.adaptive.enabled", "true")
Config.set("spark.sql.adaptive.join.enabled", "true")
Config.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
Config.set("spark.sql.sources.partitionOverwriteMode","dynamic")
Config.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")


# Config.set("spark.sql.shuffle.partitions", 100)
# Config.set("spark.default.parallelism", 200)

spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("CNPJ_Pipeline").getOrCreate()

_version_file = '2024-02'

df_raw = spark.read.parquet(f'/content/Data/raw_data/Estabelecimentos1_{_version_file}.parquet')

df_raw = df_raw.select(*[nullif(df_raw[column_name], lit('')).alias(column_name) for column_name in df_raw.columns])

df_raw = df_raw.withColumn("CNPJ", concat(df_raw["CNPJ_BASICO"],df_raw["CNPJ_ORDEM"],df_raw["CNPJ_DV"]))
df_raw = df_raw.withColumn("version_file", lit(_version_file))


df_raw = df_raw.select(

    df_raw["CNPJ"].try_cast(LongType()).alias("CNPJ"),
    df_raw["CNPJ_BASICO"].try_cast(IntegerType()).alias("CNPJ_BASICO"),
    df_raw["CNPJ_ORDEM"].try_cast(IntegerType()).alias("CNPJ_ORDEM"),
    df_raw["CNPJ_DV"].try_cast(IntegerType()).alias("CNPJ_DV"),
    df_raw["MATRIZ_FILIAL"].try_cast(IntegerType()).alias("MATRIZ_FILIAL"),
    df_raw["NOME_FANTASIA"].try_cast(StringType()).alias("NOME_FANTASIA"),
    df_raw["SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("SITUACAO_CADASTRAL"),

    to_date(nullif(df_raw["DATA_SITUACAO_CADASTRAL"],lit('0')), "yyyyMMdd").alias("DATA_SITUACAO_CADASTRAL"),

    df_raw["MOTIVO_SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("MOTIVO_SITUACAO_CADASTRAL"),
    df_raw["CIDADE_EXTERIOR"].try_cast(StringType()).alias("CIDADE_EXTERIOR"),
    df_raw["PAIS"].try_cast(StringType()).alias("PAIS"),

    to_date(df_raw["DATA_DE_INICIO_ATIVIDADE"], "yyyyMMdd").alias("DATA_DE_INICIO_ATIVIDADE"),

    df_raw["CNAE_PRINCIPAL"].try_cast(IntegerType()).alias("CNAE_PRINCIPAL"),

    split(df_raw["CNAE_SECUNDARIO"],',').alias("CNAE_SECUNDARIO"),

    df_raw["TIPO_LOGRADOURO"].try_cast(StringType()).alias("TIPO_LOGRADOURO"),
    df_raw["LOGRADOURO"].try_cast(StringType()).alias("LOGRADOURO"),
    df_raw["NUMERO"].try_cast(StringType()).alias("NUMERO"),
    df_raw["COMPLEMENTO"].try_cast(StringType()).alias("COMPLEMENTO"),
    df_raw["BAIRRO"].try_cast(StringType()).alias("BAIRRO"),
    df_raw["CEP"].try_cast(StringType()).alias("CEP"),
    df_raw["UF"].try_cast(StringType()).alias("UF"),
    df_raw["MUNICIPIO"].try_cast(StringType()).alias("MUNICIPIO"),
    df_raw["DDD"].try_cast(StringType()).alias("DDD"),
    df_raw["TELEFONE"].try_cast(StringType()).alias("TELEFONE"),
    df_raw["DDD_2"].try_cast(StringType()).alias("DDD_2"),
    df_raw["TELEFONE_2"].try_cast(StringType()).alias("TELEFONE_2"),
    df_raw["DDD_FAX"].try_cast(StringType()).alias("DDD_FAX"),
    df_raw["FAX"].try_cast(StringType()).alias("FAX"),
    df_raw["CORREIO_ELETRONICO"].try_cast(StringType()).alias("CORREIO_ELETRONICO"),
    df_raw["SITUACAO_ESPECIAL"].try_cast(StringType()).alias("SITUACAO_ESPECIAL"),
    df_raw["DATA_SITUACAO_ESPECIAL"].try_cast(StringType()).alias("DATA_SITUACAO_ESPECIAL"),
    df_raw["version_file"].try_cast(StringType()).alias("version_file"),
    current_timestamp().alias("update_date")
).where("UF = 'RJ' ")



df_raw = df_raw.na.fill({'CNPJ': -1, 'CNPJ_BASICO': -1, 'CNPJ_ORDEM': -1, 'CNPJ_DV': -1, 'MATRIZ_FILIAL': -1,'NOME_FANTASIA': 'N/A', 'SITUACAO_CADASTRAL': -1})

df_raw.repartition(1).write.partitionBy("UF").parquet(f"Data/trusted_data", mode='overwrite')

Overwriting empresas.py


### 1. Adaptive Query Execution (AQE)
O Spark, por padrão, cria um "plano de execução" antes de começar a processar os dados. O problema é que, às vezes, ele não sabe o tamanho real dos dados até começar a trabalhar. O AQE permite que o Spark mude o plano durante a execução.

`spark.sql.adaptive.enabled ("true")`
Este é o interruptor mestre. Ao ativar como true, você permite que o Spark reotimize o plano de consulta com base em estatísticas reais que ele colhe enquanto processa os dados. Ele pode, por exemplo, reduzir o número de partições se os dados forem pequenos, evitando o famoso "over-partitioning".

`spark.sql.adaptive.join.enabled ("true")`
Focado especificamente em Joins. Se o Spark perceber durante a execução que um dos lados da junção é pequeno o suficiente para caber na memória (mesmo que ele achasse que não antes), ele troca um SortMergeJoin (lento) por um BroadcastHashJoin (muito rápido) automaticamente.

### 2. Otimização de I/O
**`spark.sql.optimizer.dynamicPartitionPruning.enabled ("true")`**
O Dynamic Partition Pruning (DPP) é uma técnica inteligente para evitar ler dados desnecessários.
Imagine que você tem uma tabela gigante de "Vendas" particionada por data e uma tabela pequena de "Feriados". Se você fizer um join entre elas filtrando apenas o "Natal", o DPP faz com que o Spark leia apenas a partição do dia 25/12 na tabela de Vendas, ignorando todo o resto. Isso economiza tempo e processamento absurdos.

### 3. Performance de Serialização
**`spark.serializer ("org.apache.spark.serializer.KryoSerializer")`**
Quando o Spark precisa enviar dados pela rede (Shuffle) ou gravá-los no disco, ele precisa transformar objetos Java/Python em bytes. Isso é a serialização.
O serializador padrão do Java é muito pesado. O Kryo é muito mais compacto e rápido (chega a ser 10x mais eficiente). Usá-lo reduz o tráfego de rede e acelera o processamento de jobs complexos.

### 4. Gestão de Escrita (Overwrite)
**`spark.sql.sources.partitionOverwriteMode ("dynamic")`**
O parametro para evitar desastres.

No modo Static (padrão), se você tentar sobrescrever uma tabela particionada, o Spark apaga todas as partições da pasta antes de gravar as novas.

No modo Dynamic, o Spark apaga e sobrescreve apenas as partições que estão presentes no DataFrame que você está gravando no momento. É muito mais seguro para atualizações incrementais.



---



## Emulando o hive para explicar como funciona as Partitions

In [ ]:
%%sql
create database trusted

""


In [ ]:
%%sql
CREATE EXTERNAL TABLE spark_catalog.trusted.empresas (
 CNPJ BIGINT,
 CNPJ_BASICO INT,
 CNPJ_ORDEM INT,
 CNPJ_DV INT,
 MATRIZ_FILIAL INT,
 NOME_FANTASIA STRING,
 SITUACAO_CADASTRAL INT,
 DATA_SITUACAO_CADASTRAL DATE,
 MOTIVO_SITUACAO_CADASTRAL INT,
 CIDADE_EXTERIOR STRING,
 PAIS STRING,
 DATA_DE_INICIO_ATIVIDADE DATE,
 CNAE_PRINCIPAL INT,
 CNAE_SECUNDARIO ARRAY<STRING>,
 TIPO_LOGRADOURO STRING,
 LOGRADOURO STRING,
 NUMERO STRING,
 COMPLEMENTO STRING,
 BAIRRO STRING,
 CEP STRING,
 MUNICIPIO STRING,
 DDD STRING,
 TELEFONE STRING,
 DDD_2 STRING,
 TELEFONE_2 STRING,
 DDD_FAX STRING,
 FAX STRING,
 CORREIO_ELETRONICO STRING,
 SITUACAO_ESPECIAL STRING,
 DATA_SITUACAO_ESPECIAL STRING,
 version_file STRING,
 update_date TIMESTAMP,
 row_number INT,
 UF STRING)
USING PARQUET
PARTITIONED BY (UF)
LOCATION 'file:/content/Data/trusted_data'

""


In [ ]:
%%sql
select * from spark_catalog.trusted.empresas where UF='RJ'

CNPJ,CNPJ_BASICO,CNPJ_ORDEM,CNPJ_DV,MATRIZ_FILIAL,NOME_FANTASIA,SITUACAO_CADASTRAL,DATA_SITUACAO_CADASTRAL,MOTIVO_SITUACAO_CADASTRAL,CIDADE_EXTERIOR,PAIS,DATA_DE_INICIO_ATIVIDADE,CNAE_PRINCIPAL,CNAE_SECUNDARIO,TIPO_LOGRADOURO,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CEP,MUNICIPIO,DDD,TELEFONE,DDD_2,TELEFONE_2,DDD_FAX,FAX,CORREIO_ELETRONICO,SITUACAO_ESPECIAL,DATA_SITUACAO_ESPECIAL,version_file,update_date,row_number,UF
4010858000279,4010858,2,79,2,"COLEGIO KLUBER, CURSO KLUBER, CLUBINHO DA CRIANCA",4,2019-01-11,63,NULL,NULL,2005-04-29,8513900,NULL,RUA,BELMIRO BRAGA,SN,QUADRA 32 - LOTE 48,PARADA ANGELICA,25272050,5833,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
2552589000166,2552589,1,66,1,N/A,2,2002-02-23,0,NULL,NULL,1998-05-15,8112500,NULL,RUA,PARANHOS DA SLVA,370,NULL,ILHA DO GOVERNADOR,21931150,6001,021,3937476,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
7099012000164,7099012,1,64,1,ARCOSUR TOURS,4,2021-03-16,63,NULL,NULL,2004-11-05,7911200,"[7912100, 7990200]",RUA,RAMOM FRANCO,108,C-01,URCA,22290290,6001,21,93066747,NULL,NULL,21,22475030,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
1877360000139,1877360,1,39,1,AECA,8,2015-02-09,73,NULL,NULL,1997-04-24,9430800,"[9493600, 9499500]",RUA,D. PEDRO I,07,SALA 402,CENTRO,20060050,6001,021,2336966,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
7397562000160,7397562,1,60,1,N/A,2,2005-05-05,0,NULL,NULL,2005-05-05,6201501,"[6204000, 6209100, 8599603, 9511800]",RUA,CORONEL MADUREIRA,40,LOJA 09 - PARTE,CENTRO,28990756,5909,21,25160368,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
7397591000121,7397591,1,21,1,N/A,8,2012-12-18,1,NULL,NULL,2005-05-09,6201501,[6204000],AVENIDA,DEDO DE DEUS,611,SALA 01 PARTE,CENTRO,25940050,2907,21,25160368,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
7397731000161,7397731,1,61,1,N/A,2,2005-01-25,0,NULL,NULL,2005-01-25,8112500,NULL,AVENIDA,VIEIRA SOUTO,446,NULL,IPANEMA,22420002,6001,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
4639376000282,4639376,2,82,2,COBRART CREDITO & COBRANCA,8,2009-05-05,1,NULL,NULL,2001-07-19,8291100,NULL,AVENIDA,RIO BRANCO,147,20 ANDAR,CENTRO,20040006,6001,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
7397822000105,7397822,1,5,1,ESPACO DO SONO - PRAZER EM DORMIR.,8,2009-03-23,1,NULL,NULL,2005-05-09,4754702,NULL,ESTRADA,DOS TRES RIOS,45,LOJA A,JACAREPAGUA,22755001,6001,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ
1593295000624,1593295,6,24,2,INTERLAB,8,2007-02-16,1,NULL,NULL,2003-09-10,8640202,NULL,AVENIDA,DAS AMERICAS,7899,SALA 414,BARRA DA TIJUCA,22793081,6001,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,2024-02,2026-02-24 23:33:20.905982,NULL,RJ


In [ ]:
%%sql
REPAIR TABLE spark_catalog.trusted.empresas

""


In [ ]:
%%sql
SHOW PARTITIONS trusted.empresas

partition
UF=AC
UF=AL
UF=AM
UF=AP
UF=BA
UF=BR
UF=CE
UF=DF
UF=ES
UF=EX


# Construindo o pipeline final

In [ ]:
%%sql

with all_data as (
select
       CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
 from raw_data
union all
select
        CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
 from spark_catalog.trusted.empresas
),
rank_data as (
select
      * ,
      ROW_NUMBER() OVER ( PARTITION BY CNPJ ORDER BY update_date DESC ) AS row_number
from all_data
)

select * from rank_data
where 1=1
--and row_number = 1
and CNPJ IN (129046,264172)


CNPJ,CNPJ_BASICO,CNPJ_ORDEM,CNPJ_DV,MATRIZ_FILIAL,NOME_FANTASIA,SITUACAO_CADASTRAL,DATA_SITUACAO_CADASTRAL,MOTIVO_SITUACAO_CADASTRAL,CIDADE_EXTERIOR,PAIS,DATA_DE_INICIO_ATIVIDADE,CNAE_PRINCIPAL,CNAE_SECUNDARIO,TIPO_LOGRADOURO,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CEP,UF,MUNICIPIO,DDD,TELEFONE,DDD_2,TELEFONE_2,DDD_FAX,FAX,CORREIO_ELETRONICO,SITUACAO_ESPECIAL,DATA_SITUACAO_ESPECIAL,version_file,update_date,row_number
129046,0,1290,46,2,CARACARAI (RR),2,2005-11-03,0,NULL,NULL,1978-07-13,6422100,[6499999],RUA,OSTERNO MARREIRO DE SOUZA,S/N,NULL,SAO FRANCISCO,69360000,RR,0303,95,35321176,95,35321397,NULL,NULL,AGE1036@BB.COM.BR,NULL,NULL,2024-02,2026-02-24 23:45:26.416393,1
264172,0,2641,72,2,MARTINHO CAMPOS - MARTINHO CAMPOS (MG),2,2005-11-03,0,NULL,NULL,1982-12-20,6422100,[6499999],PRACA,GOVERNADOR VALADARES,518,NULL,CENTRO,35606000,MG,4809,NULL,NULL,NULL,NULL,NULL,NULL,CESUP.PLATBH.MG@BB.COM.BR,NULL,NULL,2024-02,2026-02-24 23:45:26.416393,1


In [ ]:
%%writefile empresas_pipeline.py
import argparse
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import time


class UpdateEmpresas:
    def __init__(self,spark_session, version_file):
        self._version_file = version_file
        self._df_raw = None
        self._df_trusted = None
        self._spark_session = spark_session

    def load_dataframe(self):
      self._df_raw = self._spark_session.read.parquet(f'/content/Data/raw_data/Estabelecimentos1_{self._version_file}.parquet')
      self._df_trusted = self._spark_session.read.parquet(f'/content/Data/trusted_data/')


    def convert_dataframe(self):
        self._df_raw = self._df_raw.select(*[nullif(self._df_raw[column_name], lit('')).alias(column_name) for column_name in self._df_raw.columns])

        self._df_raw = self._df_raw.withColumn("CNPJ", concat(self._df_raw["CNPJ_BASICO"],self._df_raw["CNPJ_ORDEM"],self._df_raw["CNPJ_DV"]))
        self._df_raw = self._df_raw.withColumn("version_file", lit(self._version_file))

        self._df_raw = self._df_raw.select(self._df_raw["CNPJ"].try_cast(LongType()).alias("CNPJ"),
                                           self._df_raw["CNPJ_BASICO"].try_cast(IntegerType()).alias("CNPJ_BASICO"),
                                           self._df_raw["CNPJ_ORDEM"].try_cast(IntegerType()).alias("CNPJ_ORDEM"),
                                           self._df_raw["CNPJ_DV"].try_cast(IntegerType()).alias("CNPJ_DV"),
                                           self._df_raw["MATRIZ_FILIAL"].try_cast(IntegerType()).alias("MATRIZ_FILIAL"),
                                           self._df_raw["NOME_FANTASIA"].try_cast(StringType()).alias("NOME_FANTASIA"),
                                           self._df_raw["SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("SITUACAO_CADASTRAL"),
                                           to_date(nullif(self._df_raw["DATA_SITUACAO_CADASTRAL"],lit('0')), "yyyyMMdd").alias("DATA_SITUACAO_CADASTRAL"),
                                           self._df_raw["MOTIVO_SITUACAO_CADASTRAL"].try_cast(IntegerType()).alias("MOTIVO_SITUACAO_CADASTRAL"),
                                           self._df_raw["CIDADE_EXTERIOR"].try_cast(StringType()).alias("CIDADE_EXTERIOR"),
                                           self._df_raw["PAIS"].try_cast(StringType()).alias("PAIS"),
                                           to_date(self._df_raw["DATA_DE_INICIO_ATIVIDADE"], "yyyyMMdd").alias("DATA_DE_INICIO_ATIVIDADE"),
                                           self._df_raw["CNAE_PRINCIPAL"].try_cast(IntegerType()).alias("CNAE_PRINCIPAL"),
                                           split(self._df_raw["CNAE_SECUNDARIO"],',').alias("CNAE_SECUNDARIO"),
                                           self._df_raw["TIPO_LOGRADOURO"].try_cast(StringType()).alias("TIPO_LOGRADOURO"),
                                           self._df_raw["LOGRADOURO"].try_cast(StringType()).alias("LOGRADOURO"),
                                           self._df_raw["NUMERO"].try_cast(StringType()).alias("NUMERO"),
                                           self._df_raw["COMPLEMENTO"].try_cast(StringType()).alias("COMPLEMENTO"),
                                           self._df_raw["BAIRRO"].try_cast(StringType()).alias("BAIRRO"),
                                           self._df_raw["CEP"].try_cast(StringType()).alias("CEP"),
                                           self._df_raw["UF"].try_cast(StringType()).alias("UF"),
                                           self._df_raw["MUNICIPIO"].try_cast(StringType()).alias("MUNICIPIO"),
                                           self._df_raw["DDD"].try_cast(StringType()).alias("DDD"),
                                           self._df_raw["TELEFONE"].try_cast(StringType()).alias("TELEFONE"),
                                           self._df_raw["DDD_2"].try_cast(StringType()).alias("DDD_2"),
                                           self._df_raw["TELEFONE_2"].try_cast(StringType()).alias("TELEFONE_2"),
                                           self._df_raw["DDD_FAX"].try_cast(StringType()).alias("DDD_FAX"),
                                           self._df_raw["FAX"].try_cast(StringType()).alias("FAX"),
                                           self._df_raw["CORREIO_ELETRONICO"].try_cast(StringType()).alias("CORREIO_ELETRONICO"),
                                           self._df_raw["SITUACAO_ESPECIAL"].try_cast(StringType()).alias("SITUACAO_ESPECIAL"),
                                           self._df_raw["DATA_SITUACAO_ESPECIAL"].try_cast(StringType()).alias("DATA_SITUACAO_ESPECIAL"),
                                           self._df_raw["version_file"].try_cast(StringType()).alias("version_file"),
                                           current_timestamp().alias("update_date")
                                           )

        self._df_raw = self._df_raw.na.fill({'CNPJ': -1, 'CNPJ_BASICO': -1, 'CNPJ_ORDEM': -1, 'CNPJ_DV': -1, 'MATRIZ_FILIAL': -1,'NOME_FANTASIA': 'N/A', 'SITUACAO_CADASTRAL': -1})
        self._df_raw.createOrReplaceTempView("raw_data")


    def emulate_hive(self):
      self._spark_session.sql("""create database trusted """)
      self._spark_session.sql("""CREATE EXTERNAL TABLE spark_catalog.trusted.empresas (
                                 CNPJ BIGINT,
                                 CNPJ_BASICO INT,
                                 CNPJ_ORDEM INT,
                                 CNPJ_DV INT,
                                 MATRIZ_FILIAL INT,
                                 NOME_FANTASIA STRING,
                                 SITUACAO_CADASTRAL INT,
                                 DATA_SITUACAO_CADASTRAL DATE,
                                 MOTIVO_SITUACAO_CADASTRAL INT,
                                 CIDADE_EXTERIOR STRING,
                                 PAIS STRING,
                                 DATA_DE_INICIO_ATIVIDADE DATE,
                                 CNAE_PRINCIPAL INT,
                                 CNAE_SECUNDARIO ARRAY<STRING>,
                                 TIPO_LOGRADOURO STRING,
                                 LOGRADOURO STRING,
                                 NUMERO STRING,
                                 COMPLEMENTO STRING,
                                 BAIRRO STRING,
                                 CEP STRING,
                                 MUNICIPIO STRING,
                                 DDD STRING,
                                 TELEFONE STRING,
                                 DDD_2 STRING,
                                 TELEFONE_2 STRING,
                                 DDD_FAX STRING,
                                 FAX STRING,
                                 CORREIO_ELETRONICO STRING,
                                 SITUACAO_ESPECIAL STRING,
                                 DATA_SITUACAO_ESPECIAL STRING,
                                 version_file STRING,
                                 update_date TIMESTAMP,
                                 row_number INT,
                                 UF STRING)
                                USING PARQUET
                                PARTITIONED BY (UF)
                                LOCATION 'file:/content/Data/trusted_data'
                                """)
      self._spark_session.sql(""" REPAIR TABLE spark_catalog.trusted.empresas """)
      self._spark_session.sql("select * from spark_catalog.trusted.empresas limit 10 ").show()


    def merge_data(self):
      self._spark_session.sql("""
      with all_data as (
                        select CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
                                from raw_data
                                union all
                        select CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
                                from spark_catalog.trusted.empresas
                        ),
           rank_data as (
                                select
                                      * ,
                                      ROW_NUMBER() OVER ( PARTITION BY CNPJ ORDER BY update_date DESC ) AS row_number
                                from all_data
                          )

                        select * from rank_data
                        where 1=1
                        and row_number = 1
                        """ ).repartition(1).write.partitionBy("UF").parquet(f"/content/Data/trusted_data", mode='overwrite')


    def execute_pipeline(self):
        self.load_dataframe()
        self.convert_dataframe()
        self.emulate_hive()
        self.merge_data()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--version', type=str,
                        help='Codigo da versao do arquivo ex: 2024-02')
    args = parser.parse_args()

    Config = SparkConf()
    Config.set("spark.sql.repl.eagerEval.enabled", True)
    Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
    Config.set("spark.sql.repl.eagerEval.truncate", "-1")
    Config.set("spark.driver.memory","10G")
    Config.set("spark.memory.fraction", 0.9)
    Config.set("spark.sql.adaptive.enabled", "true")
    Config.set("spark.sql.adaptive.join.enabled", "true")
    # Config.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
    # Config.set("spark.sql.sources.partitionOverwriteMode","dynamic")
    # Config.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")


    # Config.set("spark.sql.shuffle.partitions", 100)
    # Config.set("spark.default.parallelism", 200)

    spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("CNPJ_Pipeline").getOrCreate()

    job = UpdateEmpresas(spark, args.version)
    job.execute_pipeline()

Overwriting empresas_pipeline.py


1. O método `cache()`
Salva os dados no nível de armazenamento padrão.

Nível padrão: Para DataFrames, o padrão é o MEMORY_AND_DISK. Isso significa que o Spark tentará colocar tudo na memória RAM; se não couber, ele jogará o restante no disco para não dar erro de Out of Memory.

--
--

2. O método `persist()`


O persist() é o flexível onde é posséivel decidir como os dados serão guardados.

Você pode passar diferentes níveis de armazenamento (StorageLevel) como argumento:

*   MEMORY_ONLY: Só na RAM. Se não couber, o que ficou de fora será recalculado depois.

*   DISK_ONLY: Salva tudo direto no disco (útil para cálculos pesadíssimos que não cabem na RAM).

*   MEMORY_AND_DISK_2: Igual ao padrão, mas cria uma cópia (réplica) em outro nó do cluster por segurança.

In [ ]:
spark.sql("""
with all_data as (
select
       CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
 from raw_data
union all
select
        CNPJ, CNPJ_BASICO, CNPJ_ORDEM, CNPJ_DV, MATRIZ_FILIAL, NOME_FANTASIA, SITUACAO_CADASTRAL, DATA_SITUACAO_CADASTRAL, MOTIVO_SITUACAO_CADASTRAL, CIDADE_EXTERIOR, PAIS, DATA_DE_INICIO_ATIVIDADE, CNAE_PRINCIPAL, CNAE_SECUNDARIO, TIPO_LOGRADOURO, LOGRADOURO, NUMERO, COMPLEMENTO, BAIRRO, CEP, UF, MUNICIPIO, DDD, TELEFONE, DDD_2, TELEFONE_2, DDD_FAX, FAX, CORREIO_ELETRONICO, SITUACAO_ESPECIAL, DATA_SITUACAO_ESPECIAL, version_file, update_date
 from trusted_data
),
rank_data as (
select
      * ,
      ROW_NUMBER() OVER ( PARTITION BY CNPJ ORDER BY update_date DESC ) AS row_number
from all_data
)

select * from rank_data
where 1=1
and row_number = 1
""" ).repartition(1).write.partitionBy("UF").parquet(f"Data/trusted_data", mode='overwrite')

In [ ]:
%%sql
select version_file, count(1) from trusted_data
group by version_file
having count(1) > 1

version_file,count(1)
2024-02,4277967
2025-02,4753435


In [ ]:
df_trusted.count()

9031402

In [ ]:
df_establ.count()

4753435

## MISC

In [ ]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc   | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null   && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main"   | sudo tee /etc/apt/sources.list.d/ngrok.list   && sudo apt update   && sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com bookworm main
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 https://ngrok-agent.s3.amazonaws.com bookworm InRelease [20.3 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,626 kB]
Get:13 https://r2u.s

In [ ]:
import os
from google.colab import userdata
token = userdata.get('NGROK_TOKEN')
os.system(f"ngrok config add-authtoken {token}")

0

In [ ]:
# Assuming 'spark' is your SparkSession object
sc = spark.sparkContext
print(sc.uiWebUrl)

http://1bbe693d4ad4:4040


In [18]:
import pandas as pd

_filename = "Estabelecimentos1_2024-02"

_input_file = f"/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Aula 04/{_filename}.zip"
_output_file = f"/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Aula 04/{_filename}.parquet"

_columns = [
    "CNPJ_BASICO",
    "CNPJ_ORDEM",
    "CNPJ_DV",
    "MATRIZ_FILIAL",
    "NOME_FANTASIA",
    "SITUACAO_CADASTRAL",
    "DATA_SITUACAO_CADASTRAL",
    "MOTIVO_SITUACAO_CADASTRAL",
    "CIDADE_EXTERIOR",
    "PAIS",
    "DATA_DE_INICIO_ATIVIDADE",
    "CNAE_PRINCIPAL",
    "CNAE_SECUNDARIO",
    "TIPO_LOGRADOURO",
    "LOGRADOURO",
    "NUMERO",
    "COMPLEMENTO",
    "BAIRRO",
    "CEP",
    "UF",
    "MUNICIPIO",
    "DDD",
    "TELEFONE",
    "DDD_2",
    "TELEFONE_2",
    "DDD_FAX",
    "FAX",
    "CORREIO_ELETRONICO",
    "SITUACAO_ESPECIAL",
    "DATA_SITUACAO_ESPECIAL",
]

_df = pd.read_csv(
    _input_file,
    header=None,
    names=_columns,
    compression="zip",
    sep=";",
    encoding="iso-8859-1",
    dtype=str,
    keep_default_na=False,
)

_df.to_parquet(_output_file, index=False)

In [ ]:
df_establ = df_establ.cache().select(df_establ.columns)
size_in_bytes = df_establ._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()
df_establ.unpersist(blocking=True)
size_in_bytes = float(size_in_bytes.real)

(size_in_bytes/1024)/1024

265.78611278533936

int